# MandiMitra: Market Comparison and Price Differentials

## Objective

Identify and compare agricultural markets offering different prices
for the same vegetables across comparable reporting periods.

The analysis will examine:
- Market-level price differences.
- Comparable prices by commodity, variety and grade.
- Consistency of price advantages across dates.
- Market reporting coverage and sample reliability.
- Potential data anomalies affecting market rankings.

The analysis uses historical modal prices and does not assume
that higher historical prices guarantee better future selling returns.

In [1]:
# Imports and project setup

from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Project directories
ROOT = Path.cwd().parent
RAW_DIR = ROOT / "data" / "raw" / "parquet"

# Verify source files
RAW_FILES = sorted(RAW_DIR.glob("*.parquet"))

print("Project root:", ROOT)
print("Raw data directory:", RAW_DIR)
print("Number of Parquet files:", len(RAW_FILES))

# Create DuckDB connection
comparison_con = duckdb.connect()

# Load original annual Parquet files
comparison_con.execute(f"""
    CREATE VIEW raw_data AS
    SELECT *
    FROM read_parquet(
        '{RAW_DIR.as_posix()}/*.parquet',
        union_by_name = true
    )
""")

print("\nDuckDB connection ready.")

Project root: c:\Users\Lenovo\Documents\GrittingCode\MandiMitra
Raw data directory: c:\Users\Lenovo\Documents\GrittingCode\MandiMitra\data\raw\parquet
Number of Parquet files: 26

DuckDB connection ready.


## 1. Prepare the Market Comparison Dataset

Load the 46 confirmed vegetable commodities from the original Parquet files.

Retain valid positive modal prices, standardise text fields and dates,
and remove exact duplicate records.

The original source files will remain unchanged.

In [2]:
# Define the confirmed vegetable commodities

confirmed_vegetables = [
    "Potato", "Onion", "Tomato", "Brinjal", "Green Chilli",
    "Cauliflower", "Bhindi (Ladies Finger)", "Cabbage",
    "Bottle gourd", "Bitter gourd", "Pumpkin", "Carrot",
    "Ginger (Green)", "Garlic", "Capsicum", "Spinach",
    "Beetroot", "Peas Wet", "Drumstick", "Cowpea (Veg)",
    "Pointed gourd (Parval)", "Amaranthus",
    "French Beans (Frasbean)", "Beans", "Cluster beans",
    "Methi (Leaves)", "Elephant Yam (Suran)", "Tapioca",
    "Onion Green", "Tinda", "Sweet Potato", "Green Peas",
    "Sweet Pumpkin", "Sponge gourd", "Mint (Pudina)",
    "Indian Beans (Seam)", "Yam (Ratalu)",
    "Cowpea (Lobia/Karamani)", "Turnip",
    "Surat Beans (Papadi)", "Bunch Beans", "White Pumpkin",
    "Yam", "Duster Beans",
    "Elephant Yam (Suran)/Amorphophallus",
    "Rat Tail Radish (Mogari)"
]

# Create the comparison dataset
comparison_con.execute(f"""
    CREATE VIEW vegetable_prices AS
    SELECT DISTINCT
        TRIM(State) AS State,
        TRIM(District) AS District,
        TRIM(Market) AS Market,
        TRIM(Commodity) AS Commodity,
        TRIM(Variety) AS Variety,
        TRIM(Grade) AS Grade,
        TRY_CAST(Arrival_Date AS DATE) AS Arrival_Date,
        Min_Price,
        Max_Price,
        Modal_Price
    FROM raw_data
    WHERE TRIM(Commodity) IN ({", ".join(repr(x) for x in confirmed_vegetables)})
      AND TRY_CAST(Arrival_Date AS DATE) IS NOT NULL
      AND Modal_Price IS NOT NULL
      AND isfinite(Modal_Price)
      AND Modal_Price > 0
""")

# Verify the dataset
comparison_dataset_summary = comparison_con.execute("""
    SELECT
        COUNT(*) AS Total_Records,
        COUNT(DISTINCT Commodity) AS Commodities,
        COUNT(DISTINCT State) AS States,
        COUNT(DISTINCT District) AS Districts,
        COUNT(DISTINCT Market) AS Market_Names,
        MIN(Arrival_Date) AS First_Date,
        MAX(Arrival_Date) AS Last_Date
    FROM vegetable_prices
""").df()

display(comparison_dataset_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Total_Records,Commodities,States,Districts,Market_Names,First_Date,Last_Date
0,30336319,46,33,616,4475,2001-01-10,2026-04-20


## 2. Define the Comparison Period

The primary comparison period is January 2021 to October 2025.

November 2025 onward is excluded from the initial comparison because
the dataset shows a substantial decline in reporting volume during
that period.

Markets will be identified using State, District and Market together
to avoid treating similarly named markets in different locations
as the same market.

In [3]:
# Define the historical comparison period

COMPARISON_START = "2021-01-01"
COMPARISON_END = "2025-10-31"

comparison_period_summary = comparison_con.execute("""
    SELECT
        COUNT(*) AS Total_Records,
        COUNT(DISTINCT Commodity) AS Commodities,
        COUNT(DISTINCT State || '|' || District || '|' || Market)
            AS Unique_Markets,
        MIN(Arrival_Date) AS First_Date,
        MAX(Arrival_Date) AS Last_Date
    FROM vegetable_prices
    WHERE Arrival_Date BETWEEN
        DATE '2021-01-01' AND DATE '2025-10-31'
""").df()

display(comparison_period_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Total_Records,Commodities,Unique_Markets,First_Date,Last_Date
0,11412781,45,2222,2021-01-01,2025-10-31


## 3. Build Comparable Market-Day Prices

Create one observation for each unique combination of:

- Commodity
- Variety
- Grade
- Arrival date
- Market identity (State, District and Market)

If multiple records exist for the same combination within a market,
use their median modal price.

This creates a consistent basis for comparing prices across markets
without mixing different varieties, grades or reporting dates.

In [4]:
# Build market-day price observations

comparison_con.execute("""
    CREATE VIEW market_day_prices AS
    SELECT
        Commodity,
        Variety,
        Grade,
        Arrival_Date,
        State,
        District,
        Market,
        MEDIAN(Modal_Price) AS Median_Modal_Price,
        COUNT(*) AS Source_Records
    FROM vegetable_prices
    WHERE Arrival_Date BETWEEN
        DATE '2021-01-01' AND DATE '2025-10-31'
    GROUP BY
        Commodity,
        Variety,
        Grade,
        Arrival_Date,
        State,
        District,
        Market
""")

# Check the resulting comparison dataset
market_day_summary = comparison_con.execute("""
    SELECT
        COUNT(*) AS Market_Day_Observations,
        COUNT(DISTINCT Commodity) AS Commodities,
        COUNT(DISTINCT State || '|' || District || '|' || Market)
            AS Unique_Markets,
        MEDIAN(Source_Records) AS Median_Source_Records,
        MAX(Source_Records) AS Maximum_Source_Records
    FROM market_day_prices
""").df()

display(market_day_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Market_Day_Observations,Commodities,Unique_Markets,Median_Source_Records,Maximum_Source_Records
0,11412781,45,2222,1.0,1


## 4. Measure Cross-Market Comparability

For each commodity, variety, grade and arrival date, count how many
different markets reported prices.

This identifies observations that can support meaningful
same-day comparisons between markets.

In [5]:
# Count markets reporting the same commodity, variety, grade and date

comparable_groups = comparison_con.execute("""
    SELECT
        Commodity,
        Variety,
        Grade,
        Arrival_Date,
        COUNT(*) AS Reporting_Markets
    FROM market_day_prices
    GROUP BY
        Commodity,
        Variety,
        Grade,
        Arrival_Date
""").df()

# Summarise the availability of comparable market groups

comparability_summary = pd.DataFrame({
    "Metric": [
        "Total commodity-variety-grade-date groups",
        "Groups with 2+ reporting markets",
        "Groups with 3+ reporting markets",
        "Groups with 5+ reporting markets",
        "Groups with 10+ reporting markets"
    ],
    "Count": [
        len(comparable_groups),
        (comparable_groups["Reporting_Markets"] >= 2).sum(),
        (comparable_groups["Reporting_Markets"] >= 3).sum(),
        (comparable_groups["Reporting_Markets"] >= 5).sum(),
        (comparable_groups["Reporting_Markets"] >= 10).sum()
    ]
})

comparability_summary["Percentage"] = (
    comparability_summary["Count"] /
    len(comparable_groups) * 100
).round(2)

display(comparability_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Metric,Count,Percentage
0,Total commodity-variety-grade-date groups,260607,100.00
1,Groups with 2+ reporting markets,225722,86.61
2,Groups with 3+ reporting markets,207945,79.79
3,Groups with 5+ reporting markets,188013,72.14
4,Groups with 10+ reporting markets,160769,61.69


## 5. Comparability by Commodity

Measure the availability of same-day market comparisons separately
for each vegetable.

This identifies commodities with sufficient market coverage and
those that may require more restrictive analysis.

In [6]:
# Summarise cross-market comparability by commodity

commodity_comparability = (
    comparable_groups
    .groupby("Commodity")
    .agg(
        Total_Groups=("Reporting_Markets", "size"),
        Median_Reporting_Markets=("Reporting_Markets", "median"),
        Groups_2_Markets=(
            "Reporting_Markets",
            lambda x: (x >= 2).sum()
        ),
        Groups_3_Markets=(
            "Reporting_Markets",
            lambda x: (x >= 3).sum()
        ),
        Groups_5_Markets=(
            "Reporting_Markets",
            lambda x: (x >= 5).sum()
        )
    )
    .reset_index()
)

commodity_comparability["Pct_Groups_3_Markets"] = (
    commodity_comparability["Groups_3_Markets"] /
    commodity_comparability["Total_Groups"] * 100
).round(2)

commodity_comparability["Pct_Groups_5_Markets"] = (
    commodity_comparability["Groups_5_Markets"] /
    commodity_comparability["Total_Groups"] * 100
).round(2)

commodity_comparability = commodity_comparability.sort_values(
    "Total_Groups",
    ascending=False
)

display(commodity_comparability)

,Commodity,Total_Groups,Median_Reporting_Markets,Groups_2_Markets,Groups_3_Markets,Groups_5_Markets,Pct_Groups_3_Markets,Pct_Groups_5_Markets
26,Onion,32710,6.0,24538,19922,17399,60.90,53.19
30,Potato,28839,9.0,24721,21781,17628,75.53,61.13
40,Tomato,13043,73.0,11801,11742,11620,90.03,89.09
19,Garlic,11177,23.0,8466,7988,7764,71.47,69.46
6,Brinjal,10628,78.0,10081,9955,9870,93.67,92.87
11,Cauliflower,9785,24.0,8351,8205,7993,83.85,81.69
36,Sweet Potato,6390,6.0,5571,4831,3688,75.60,57.72
22,Green Peas,5940,5.0,4710,4186,3139,70.47,52.85
10,Carrot,5637,64.0,4841,4574,4380,81.14,77.70
1,Beans,5515,13.0,4633,4357,4066,79.00,73.73


## 6. Market Participation Across Comparison Dates

For each commodity-variety-grade-date group, examine the number
of reporting markets.

Summarise the median, 25th percentile and 75th percentile of
market participation for each commodity.

This helps identify whether market comparisons are generally
broad-based or depend on a small number of reporting markets.

In [7]:
# Summarise market participation by commodity

market_participation = (
    comparable_groups
    .groupby("Commodity")
    .agg(
        Comparison_Groups=("Reporting_Markets", "size"),
        Median_Markets=("Reporting_Markets", "median"),
        P25_Markets=(
            "Reporting_Markets",
            lambda x: x.quantile(0.25)
        ),
        P75_Markets=(
            "Reporting_Markets",
            lambda x: x.quantile(0.75)
        ),
        Maximum_Markets=("Reporting_Markets", "max")
    )
    .reset_index()
    .sort_values("Comparison_Groups", ascending=False)
)

display(market_participation)

,Commodity,Comparison_Groups,Median_Markets,P25_Markets,P75_Markets,Maximum_Markets
26,Onion,32710,6.0,2.0,25.00,262
30,Potato,28839,9.0,3.0,46.00,270
40,Tomato,13043,73.0,22.0,106.00,246
19,Garlic,11177,23.0,2.0,48.00,111
6,Brinjal,10628,78.0,16.0,155.00,245
11,Cauliflower,9785,24.0,10.0,108.00,247
36,Sweet Potato,6390,6.0,3.0,10.00,87
22,Green Peas,5940,5.0,2.0,11.00,59
10,Carrot,5637,64.0,13.0,117.00,226
1,Beans,5515,13.0,4.0,19.00,198


## 7. Create the Matched Market Comparison Dataset

Retain commodity-variety-grade-date groups with at least three
reporting markets.

Calculate the median price across reporting markets for each group.
Then calculate each market's percentage premium or discount
relative to that same-day reference price.

Positive values indicate prices above the peer-market median.
Negative values indicate prices below the peer-market median.

Pothencode observations from 2023–2024 are excluded from this
primary ranking analysis because of previously identified
source-price anomalies. Original records remain unchanged.

In [8]:
# Create matched market comparisons

comparison_con.execute("""
    CREATE VIEW matched_market_prices AS
    WITH eligible_groups AS (
        SELECT
            Commodity,
            Variety,
            Grade,
            Arrival_Date
        FROM market_day_prices
        GROUP BY
            Commodity,
            Variety,
            Grade,
            Arrival_Date
        HAVING COUNT(*) >= 3
    ),

    filtered_prices AS (
        SELECT *
        FROM market_day_prices
        WHERE NOT (
            LOWER(Market) = 'pothencode'
            AND YEAR(Arrival_Date) IN (2023, 2024)
        )
    ),

    matched_prices AS (
        SELECT p.*
        FROM filtered_prices p
        INNER JOIN eligible_groups g
        USING (Commodity, Variety, Grade, Arrival_Date)
    ),

    reference_prices AS (
        SELECT
            Commodity,
            Variety,
            Grade,
            Arrival_Date,
            MEDIAN(Median_Modal_Price) AS Peer_Median_Price,
            COUNT(*) AS Reporting_Markets
        FROM matched_prices
        GROUP BY
            Commodity,
            Variety,
            Grade,
            Arrival_Date
        HAVING COUNT(*) >= 3
    )

    SELECT
        p.*,
        r.Peer_Median_Price,
        r.Reporting_Markets,
        100.0 * (
            p.Median_Modal_Price / r.Peer_Median_Price - 1
        ) AS Price_Premium_Pct

    FROM matched_prices p
    INNER JOIN reference_prices r
    USING (Commodity, Variety, Grade, Arrival_Date)
""")

matched_summary = comparison_con.execute("""
    SELECT
        COUNT(*) AS Matched_Observations,
        COUNT(DISTINCT Commodity) AS Commodities,
        COUNT(DISTINCT State || '|' || District || '|' || Market)
            AS Unique_Markets,
        COUNT(DISTINCT Commodity || '|' || Variety || '|' || Grade)
            AS Commodity_Variety_Grade_Combinations,
        MIN(Arrival_Date) AS First_Date,
        MAX(Arrival_Date) AS Last_Date
    FROM matched_market_prices
""").df()

display(matched_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Matched_Observations,Commodities,Unique_Markets,Commodity_Variety_Grade_Combinations,First_Date,Last_Date
0,11331685,45,2213,283,2021-01-01,2025-10-31


## 8. Market Price Premiums and Discounts

Summarise each market's price differences relative to the
same-day median of comparable markets.

A positive median premium indicates that a market generally
reported higher prices than its peers.

A negative median premium indicates generally lower prices.

Also measure the percentage of observations above the peer
median and the median absolute deviation to understand consistency.

In [9]:
# Calculate market-level price comparison metrics

market_price_performance = comparison_con.execute("""
    SELECT
        Commodity,
        State,
        District,
        Market,

        COUNT(*) AS Matched_Observations,
        COUNT(DISTINCT Arrival_Date) AS Comparison_Days,

        MEDIAN(Price_Premium_Pct) AS Median_Premium_Pct,

        100.0 * AVG(
            CASE WHEN Price_Premium_Pct > 0 THEN 1.0 ELSE 0.0 END
        ) AS Pct_Days_Above_Peer_Median,

        MEDIAN(ABS(Price_Premium_Pct)) AS Median_Absolute_Deviation,

        MEDIAN(Median_Modal_Price) AS Median_Market_Price

    FROM matched_market_prices
    GROUP BY
        Commodity,
        State,
        District,
        Market
""").df()

# Round percentage metrics for readability

percentage_columns = [
    "Median_Premium_Pct",
    "Pct_Days_Above_Peer_Median",
    "Median_Absolute_Deviation"
]

market_price_performance[percentage_columns] = (
    market_price_performance[percentage_columns].round(2)
)

display(
    market_price_performance
    .sort_values("Matched_Observations", ascending=False)
    .head(20)
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Commodity,State,District,Market,Matched_Observations,Comparison_Days,Median_Premium_Pct,Pct_Days_Above_Peer_Median,Median_Absolute_Deviation,Median_Market_Price
1656,Onion,Kerala,Alappuzha,Kayamkulam,3203,1653,-11.36,11.43,12.15,3200.0
14786,Potato,West Bengal,Paschim Bardhaman,Durgapur,3148,1631,2.40,56.23,7.56,1400.0
16515,Onion,Kerala,Kottayam,Kanjirappally,2871,1451,40.74,88.75,40.74,4300.0
14017,Brinjal,Karnataka,Bangalore,Ramanagara,2723,1639,19.48,65.37,31.15,2000.0
12380,Onion,Kerala,Palakad,Palakkad,2673,1354,12.50,86.16,12.88,4300.0
7450,Onion,Kerala,Palakad,Pattambi,2647,1359,-2.70,29.96,6.38,3800.0
21461,Amaranthus,Kerala,Palakad,Palakkad,2580,1319,-6.67,28.10,12.50,2600.0
59,Onion,Kerala,Alappuzha,Chengannur,2565,1288,23.08,80.90,23.33,4000.0
1748,Brinjal,Karnataka,Chamrajnagar,Chamaraj Nagar,2507,1351,-43.75,12.88,45.83,1000.0
21481,Brinjal,Kerala,Kottayam,Thalayolaparambu,2495,1257,66.67,95.19,66.67,3000.0


In [10]:
# Rename the metric to describe what it actually measures
market_price_performance = market_price_performance.rename(
    columns={
        "Median_Absolute_Deviation": "Median_Absolute_Premium_Pct"
    }
)

# Examine the amount of evidence available for each market
market_evidence_summary = (
    market_price_performance
    .groupby("Commodity")
    .agg(
        Eligible_Markets=("Market", "count"),
        Median_Observations=("Matched_Observations", "median"),
        P25_Observations=("Matched_Observations", lambda x: x.quantile(0.25)),
        P75_Observations=("Matched_Observations", lambda x: x.quantile(0.75)),
        Median_Comparison_Days=("Comparison_Days", "median"),
        Markets_100_Obs_30_Days=(
            "Matched_Observations",
            lambda x: (x >= 100).sum()
        ),
        Markets_250_Obs=(
            "Matched_Observations",
            lambda x: (x >= 250).sum()
        )
    )
    .reset_index()
)

display(
    market_evidence_summary
    .sort_values("Eligible_Markets", ascending=False)
    .round(1)
)

,Commodity,Eligible_Markets,Median_Observations,P25_Observations,P75_Observations,Median_Comparison_Days,Markets_100_Obs_30_Days,Markets_250_Obs
40,Tomato,1579,444.0,98.5,1069.5,444.0,1183,1025
26,Onion,1540,440.5,120.8,1110.2,439.0,1178,1036
6,Brinjal,1508,432.0,74.2,1036.2,432.0,1095,956
30,Potato,1460,513.0,136.0,1174.2,505.5,1143,1012
21,Green Chilli,1401,444.0,54.0,1006.0,444.0,999,872
3,Bhindi (Ladies Finger),1385,396.0,53.0,669.0,396.0,966,806
4,Bitter gourd,1318,270.0,36.2,544.0,270.0,825,671
8,Cabbage,1281,353.0,49.0,682.0,353.0,880,721
11,Cauliflower,1260,394.0,68.5,736.5,394.0,890,743
5,Bottle gourd,1142,423.5,34.0,818.8,423.5,751,652


In [11]:
# Minimum evidence required for primary market rankings
MIN_MATCHED_OBSERVATIONS = 100
MIN_COMPARISON_DAYS = 30

rankable_markets = market_price_performance[
    (market_price_performance["Matched_Observations"] >= MIN_MATCHED_OBSERVATIONS)
    & (market_price_performance["Comparison_Days"] >= MIN_COMPARISON_DAYS)
].copy()

# Summarise how many markets pass the threshold for each vegetable
rankable_market_summary = (
    rankable_markets
    .groupby("Commodity")
    .agg(
        Rankable_Markets=("Market", "count"),
        Median_Observations=("Matched_Observations", "median"),
        Median_Comparison_Days=("Comparison_Days", "median")
    )
    .reset_index()
    .sort_values("Rankable_Markets", ascending=False)
)

display(rankable_market_summary.round(1))

print("Total rankable market-commodity combinations:",
      len(rankable_markets))

,Commodity,Rankable_Markets,Median_Observations,Median_Comparison_Days
39,Tomato,1183,740.0,726.0
26,Onion,1178,732.0,702.5
30,Potato,1143,808.0,801.0
6,Brinjal,1095,713.0,713.0
21,Green Chilli,999,673.0,673.0
3,Bhindi (Ladies Finger),966,473.0,473.0
11,Cauliflower,890,540.5,537.5
8,Cabbage,880,458.0,455.0
4,Bitter gourd,825,446.0,446.0
5,Bottle gourd,751,535.0,535.0


Total rankable market-commodity combinations: 16950


In [12]:
# Rank markets within each commodity
ranked_markets = rankable_markets.copy()

ranked_markets["Premium_Rank"] = (
    ranked_markets
    .groupby("Commodity")["Median_Premium_Pct"]
    .rank(method="min", ascending=False)
)

ranked_markets["Consistency_Rank"] = (
    ranked_markets
    .groupby("Commodity")["Pct_Days_Above_Peer_Median"]
    .rank(method="min", ascending=False)
)

# Show the five highest-premium markets for each commodity
top_markets = (
    ranked_markets
    .sort_values(
        ["Commodity", "Median_Premium_Pct"],
        ascending=[True, False]
    )
    .groupby("Commodity")
    .head(5)
    [[
        "Commodity", "State", "District", "Market",
        "Matched_Observations", "Comparison_Days",
        "Median_Premium_Pct",
        "Pct_Days_Above_Peer_Median",
        "Median_Absolute_Premium_Pct"
    ]]
)

display(top_markets.round(2).reset_index(drop=True))

,Commodity,State,District,Market,Matched_Observations,Comparison_Days,Median_Premium_Pct,Pct_Days_Above_Peer_Median,Median_Absolute_Premium_Pct
0,Amaranthus,Kerala,Thiruvananthapuram,Balarampuram,216,216,150.00,99.07,150.00
1,Amaranthus,Tamil Nadu,The Nilgiris,Coonoor (Uzhavar Sandhai ),443,443,140.00,100.00,140.00
2,Amaranthus,Tamil Nadu,Krishnagiri,Hosur (Uzhavar Sandhai ),445,445,140.00,100.00,140.00
3,Amaranthus,Tamil Nadu,The Nilgiris,Gudalur (Uzhavar Sandhai ),158,158,140.00,100.00,140.00
4,Amaranthus,Tamil Nadu,The Nilgiris,Udhagamandalam (Uzhavar Sandhai ),444,444,140.00,100.00,140.00
...,...,...,...,...,...,...,...,...,...
206,Yam (Ratalu),Tamil Nadu,Dharmapuri,Dharmapuri (Uzhavar Sandhai ),375,375,64.29,90.40,64.29
207,Yam (Ratalu),Tamil Nadu,Coimbatore,Udumalpet,249,249,62.50,97.59,62.50
208,Yam (Ratalu),Nagaland,Mokokchung,Tuli,103,103,60.78,84.47,60.78
209,Yam (Ratalu),Gujarat,Surat,Surat,1078,1078,52.45,85.25,52.45


In [13]:
# Identify possible farmers' markets in the shortlist
top_markets = top_markets.copy()

top_markets["Possible_Farmers_Market"] = (
    top_markets["Market"]
    .str.contains(r"uzhavar|sandhai|santhai|rythu bazar",
                  case=False, na=False)
)

print("Total shortlisted markets:", len(top_markets))
print("Possible farmers' market entries:",
      top_markets["Possible_Farmers_Market"].sum())
print("Entries with premium >= 50%:",
      (top_markets["Median_Premium_Pct"] >= 50).sum())

# Inspect the six major vegetables
major_vegetables = [
    "Potato", "Onion", "Tomato",
    "Brinjal", "Green Chilli", "Cauliflower"
]

major_market_shortlist = (
    top_markets[
        top_markets["Commodity"].isin(major_vegetables)
    ]
    .sort_values(
        ["Commodity", "Median_Premium_Pct"],
        ascending=[True, False]
    )
)

display(
    major_market_shortlist[
        [
            "Commodity", "State", "District", "Market",
            "Possible_Farmers_Market",
            "Matched_Observations", "Comparison_Days",
            "Median_Premium_Pct",
            "Pct_Days_Above_Peer_Median"
        ]
    ].round(2).reset_index(drop=True)
)

Total shortlisted markets: 211
Possible farmers' market entries: 16
Entries with premium >= 50%: 180


,Commodity,State,District,Market,Possible_Farmers_Market,Matched_Observations,Comparison_Days,Median_Premium_Pct,Pct_Days_Above_Peer_Median
0,Brinjal,Andaman and Nicobar,South Andaman,Port Blair,False,133,133,370.59,100.00
1,Brinjal,Kerala,Thiruvananthapuram,Parassala,False,1602,1103,237.70,99.94
2,Brinjal,Telangana,Adilabad,Adilabad (Rythu Bazar),True,586,586,212.50,99.83
3,Brinjal,Kerala,Thiruvananthapuram,Nedumangadu,False,575,575,211.11,100.00
4,Brinjal,Kerala,Ernakulam,Thrippunithura,False,1293,1293,202.22,100.00
5,Cauliflower,Telangana,Adilabad,Adilabad (Rythu Bazar),True,644,644,233.33,100.00
6,Cauliflower,Odisha,Kalahandi,Kesinga,False,398,398,225.00,100.00
7,Cauliflower,Nagaland,Kohima,Jalukie,False,103,103,218.18,100.00
8,Cauliflower,Kerala,Thiruvananthapuram,Parassala,False,141,141,212.00,100.00
9,Cauliflower,Kerala,Thiruvananthapuram,Aralamoodu,False,1115,1115,204.18,100.00


In [14]:
# Investigate markets with unusually high historical premiums
extreme_market_check = comparison_con.execute("""
    SELECT
        Commodity,
        State,
        District,
        Market,
        COUNT(*) AS Matched_Observations,
        MEDIAN(Median_Modal_Price) AS Median_Market_Price,
        MEDIAN(Peer_Median_Price) AS Median_Peer_Price,
        MEDIAN(Price_Premium_Pct) AS Median_Premium_Pct,
        MIN(Arrival_Date) AS First_Date,
        MAX(Arrival_Date) AS Last_Date,
        COUNT(DISTINCT YEAR(Arrival_Date)) AS Years_Observed
    FROM matched_market_prices
    WHERE Commodity IN (
        'Potato', 'Onion', 'Tomato',
        'Brinjal', 'Green Chilli', 'Cauliflower'
    )
    GROUP BY Commodity, State, District, Market
    HAVING MEDIAN(Price_Premium_Pct) >= 100
    ORDER BY Median_Premium_Pct DESC
    LIMIT 30
""").df()

display(extreme_market_check.round(2))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_23860\3443406012.py:26: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(extreme_market_check.round(2))


,Commodity,State,District,Market,Matched_Observations,Median_Market_Price,Median_Peer_Price,Median_Premium_Pct,First_Date,Last_Date,Years_Observed
0,Green Chilli,Kerala,Kollam,Valakom VFPCK,1,3950000.0,4100.0,96241.46,2024-07-05,2024-07-05,1
1,Cauliflower,Rajasthan,Sanchore,Sanchore,2,220000.0,2325.0,9318.60,2024-11-11,2024-12-05,1
2,Potato,Meghalaya,East Khasi Hills,Sohra,43,36000.0,1400.0,2900.00,2022-10-01,2024-08-03,3
3,Tomato,Meghalaya,East Khasi Hills,Sohra,26,39000.0,1530.0,2431.37,2022-10-01,2023-10-12,2
4,Potato,Kerala,Thiruvananthapuram,Pothencode,135,22540.0,1000.0,1644.52,2021-01-02,2025-04-03,2
5,Onion,Kerala,Thiruvananthapuram,Pothencode,135,40040.0,2500.0,1440.00,2021-01-02,2025-04-03,2
6,Brinjal,Meghalaya,East Khasi Hills,Sohra,52,34000.0,2000.0,1426.67,2022-01-10,2024-08-03,3
7,Cauliflower,Andaman and Nicobar,Nicobar,Car Nicobar,26,18000.0,1250.0,1342.31,2022-02-24,2023-09-07,2
8,Cauliflower,Bihar,Madhubani,Jainagar,4,49000.0,3500.0,1280.56,2023-07-14,2023-07-18,1
9,Green Chilli,Kerala,Thirssur,KADUKUTTY VFPCK,1,32000.0,2725.0,1074.31,2025-07-03,2025-07-03,1


In [15]:
# Inspect annual price distributions for suspicious markets

suspicious_markets = [
    ("Green Chilli", "Kerala", "Kollam", "Valakom VFPCK"),
    ("Cauliflower", "Rajasthan", "Sanchore", "Sanchore"),
    ("Potato", "Meghalaya", "East Khasi Hills", "Sohra"),
    ("Tomato", "Meghalaya", "East Khasi Hills", "Sohra"),
    ("Brinjal", "Meghalaya", "East Khasi Hills", "Sohra"),
    ("Potato", "Kerala", "Thiruvananthapuram", "Pothencode"),
    ("Onion", "Kerala", "Thiruvananthapuram", "Pothencode"),
    ("Green Chilli", "Andaman and Nicobar", "South Andaman", "Port Blair"),
]

suspicious_df = pd.DataFrame(
    suspicious_markets,
    columns=["Commodity", "State", "District", "Market"]
)

comparison_con.register("suspicious_markets", suspicious_df)

suspicious_price_history = comparison_con.execute("""
    SELECT
        p.Commodity,
        p.State,
        p.District,
        p.Market,
        YEAR(p.Arrival_Date) AS Year,
        COUNT(*) AS Observations,
        MEDIAN(p.Median_Modal_Price) AS Median_Price,
        QUANTILE_CONT(p.Median_Modal_Price, 0.25) AS Q1_Price,
        QUANTILE_CONT(p.Median_Modal_Price, 0.75) AS Q3_Price,
        MIN(p.Median_Modal_Price) AS Minimum_Price,
        MAX(p.Median_Modal_Price) AS Maximum_Price
    FROM market_day_prices p
    INNER JOIN suspicious_markets s
        USING (Commodity, State, District, Market)
    GROUP BY
        p.Commodity, p.State, p.District,
        p.Market, YEAR(p.Arrival_Date)
    ORDER BY
        p.Commodity, p.Market, Year
""").df()

display(suspicious_price_history.round(2))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Commodity,State,District,Market,Year,Observations,Median_Price,Q1_Price,Q3_Price,Minimum_Price,Maximum_Price
0,Brinjal,Meghalaya,East Khasi Hills,Sohra,2022,10,40000.0,40000.0,43000.0,40000.0,46000.0
1,Brinjal,Meghalaya,East Khasi Hills,Sohra,2023,19,38000.0,34000.0,43000.0,33000.0,82000.0
2,Brinjal,Meghalaya,East Khasi Hills,Sohra,2024,23,3000.0,3000.0,4000.0,3000.0,10000.0
3,Cauliflower,Rajasthan,Sanchore,Sanchore,2024,2,220000.0,205000.0,235000.0,190000.0,250000.0
4,Green Chilli,Andaman and Nicobar,South Andaman,Port Blair,2022,93,25000.0,25000.0,25000.0,7500.0,30000.0
5,Green Chilli,Andaman and Nicobar,South Andaman,Port Blair,2023,27,30000.0,30000.0,30000.0,25000.0,30000.0
6,Green Chilli,Andaman and Nicobar,South Andaman,Port Blair,2024,10,30000.0,30000.0,30000.0,30000.0,30000.0
7,Green Chilli,Kerala,Kollam,Valakom VFPCK,2024,1,3950000.0,3950000.0,3950000.0,3950000.0,3950000.0
8,Onion,Kerala,Thiruvananthapuram,Pothencode,2021,59,2900.0,2700.0,3750.0,2300.0,5200.0
9,Onion,Kerala,Thiruvananthapuram,Pothencode,2023,220,10440.0,9040.0,19540.0,4260.0,39040.0


In [16]:
# Identify market-level annual price-scale changes

annual_market_prices = comparison_con.execute("""
    SELECT
        Commodity,
        State,
        District,
        Market,
        YEAR(Arrival_Date) AS Year,
        COUNT(*) AS Observations,
        MEDIAN(Median_Modal_Price) AS Annual_Median_Price
    FROM market_day_prices
    GROUP BY
        Commodity, State, District, Market,
        YEAR(Arrival_Date)
    HAVING COUNT(*) >= 10
""").df()

annual_market_prices = annual_market_prices.sort_values(
    ["Commodity", "State", "District", "Market", "Year"]
)

annual_market_prices["Previous_Year"] = (
    annual_market_prices
    .groupby(["Commodity", "State", "District", "Market"])["Year"]
    .shift(1)
)

annual_market_prices["Previous_Annual_Median"] = (
    annual_market_prices
    .groupby(["Commodity", "State", "District", "Market"])["Annual_Median_Price"]
    .shift(1)
)

annual_market_prices["Yearly_Price_Ratio"] = (
    annual_market_prices["Annual_Median_Price"]
    / annual_market_prices["Previous_Annual_Median"]
)

annual_market_prices["Yearly_Price_Change_Pct"] = (
    (annual_market_prices["Yearly_Price_Ratio"] - 1) * 100
)

large_price_shifts = annual_market_prices[
    (annual_market_prices["Previous_Year"] == annual_market_prices["Year"] - 1)
    & (
        (annual_market_prices["Yearly_Price_Ratio"] >= 5)
        | (annual_market_prices["Yearly_Price_Ratio"] <= 0.2)
    )
].copy()

print("Annual market records:", len(annual_market_prices))
print("Large consecutive-year price shifts:", len(large_price_shifts))

display(
    large_price_shifts
    .sort_values(
        "Yearly_Price_Ratio",
        key=lambda x: np.maximum(x, 1 / x),
        ascending=False
    )
    .head(25)
    .round(2)
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Annual market records: 70263
Large consecutive-year price shifts: 160


,Commodity,State,District,Market,Year,Observations,Annual_Median_Price,Previous_Year,Previous_Annual_Median,Yearly_Price_Ratio,Yearly_Price_Change_Pct
54600,Bitter gourd,Kerala,Kottayam,Kattampak VFPCK,2025,10,4500.0,2024.0,29.0,155.17,15417.24
17898,Bitter gourd,Kerala,Kottayam,Thottuva VFPCK,2025,24,5350.0,2024.0,38.0,140.79,13978.95
4399,Methi (Leaves),Maharashtra,Pune,Khed,2024,237,1500.0,2023.0,11.0,136.36,13536.36
12101,Tapioca,Kerala,Kottayam,Thottuva VFPCK,2025,21,2000.0,2024.0,15.0,133.33,13233.33
69935,Spinach,Maharashtra,Ahmednagar,Ahmednagar,2024,99,1600.0,2023.0,13.0,123.08,12207.69
5248,Methi (Leaves),Maharashtra,Ahmednagar,Ahmednagar,2024,117,2000.0,2023.0,18.0,111.11,11011.11
57915,Bottle gourd,Kerala,Kottayam,Adithyapuram VFPCK,2025,15,1500.0,2024.0,16.0,93.75,9275.00
47789,Cowpea (Veg),Kerala,Kottayam,Adithyapuram VFPCK,2025,147,5300.0,2024.0,57.0,92.98,9198.25
31368,Bitter gourd,Kerala,Kottayam,Adithyapuram VFPCK,2025,96,6500.0,2024.0,78.0,83.33,8233.33
70222,Indian Beans (Seam),Odisha,Bolangir,Kantabaji,2023,30,5000.0,2022.0,400000.0,0.01,-98.75


In [17]:
# Check whether large price shifts affect multiple commodities
# within the same market and year

market_year_anomaly_summary = (
    large_price_shifts
    .assign(
        Shift_Direction=lambda x: np.where(
            x["Yearly_Price_Ratio"] > 1,
            "Large increase",
            "Large decrease"
        )
    )
    .groupby(
        ["State", "District", "Market", "Year"]
    )
    .agg(
        Affected_Commodities=("Commodity", "nunique"),
        Commodity_List=(
            "Commodity",
            lambda x: ", ".join(sorted(x.unique()))
        ),
        Affected_Series=("Commodity", "count"),
        Median_Price_Ratio=("Yearly_Price_Ratio", "median"),
        Minimum_Ratio=("Yearly_Price_Ratio", "min"),
        Maximum_Ratio=("Yearly_Price_Ratio", "max"),
        Direction=("Shift_Direction", "first")
    )
    .reset_index()
    .sort_values(
        ["Affected_Commodities", "Affected_Series"],
        ascending=False
    )
)

print(
    "Market-year groups with large price shifts:",
    len(market_year_anomaly_summary)
)

print(
    "Groups affecting 3+ commodities:",
    (market_year_anomaly_summary["Affected_Commodities"] >= 3).sum()
)

display(market_year_anomaly_summary.head(30).round(2))

Market-year groups with large price shifts: 107
Groups affecting 3+ commodities: 8


,State,District,Market,Year,Affected_Commodities,Commodity_List,Affected_Series,Median_Price_Ratio,Minimum_Ratio,Maximum_Ratio,Direction
30,Kerala,Palakad,Kollengode,2023,15,"Amaranthus, Beetroot, Bhindi (Ladies Finger), ...",15,0.11,0.07,0.19,Large decrease
34,Kerala,Thiruvananthapuram,Pothencode,2025,14,"Amaranthus, Beetroot, Brinjal, Cabbage, Capsic...",14,0.17,0.11,0.19,Large decrease
55,Meghalaya,East Khasi Hills,Sohra,2024,5,"Bitter gourd, Brinjal, Cabbage, Carrot, Potato",5,0.08,0.06,0.14,Large decrease
31,Kerala,Palakad,Kollengode,2024,4,"Cabbage, Carrot, Potato, Pumpkin",4,0.09,0.07,0.11,Large decrease
93,Uttar Pradesh,Chitrakut,Karvi,2023,4,"Bottle gourd, Cabbage, Carrot, Cauliflower",4,0.10,0.08,0.15,Large decrease
19,Karnataka,Kalburgi,Kalburgi,2022,3,"Bhindi (Ladies Finger), Capsicum, Green Chilli",3,6.60,5.28,11.61,Large increase
27,Kerala,Kottayam,Adithyapuram VFPCK,2025,3,"Bitter gourd, Bottle gourd, Cowpea (Veg)",3,92.98,83.33,93.75,Large increase
29,Kerala,Kottayam,Thottuva VFPCK,2025,3,"Bitter gourd, Pumpkin, Tapioca",3,133.33,55.56,140.79,Large increase
20,Karnataka,Kolar,Chintamani,2022,2,"Cabbage, Cauliflower",2,6.61,6.00,7.22,Large increase
24,Kerala,Idukki,Vandiperiyar,2024,2,"Garlic, Ginger (Green)",2,6.78,5.00,8.56,Large increase


In [18]:
# Inspect commodity-level details for market-years
# with large shifts across at least three commodities

multi_commodity_anomalies = market_year_anomaly_summary[
    market_year_anomaly_summary["Affected_Commodities"] >= 3
][
    ["State", "District", "Market", "Year"]
]

detailed_anomaly_check = large_price_shifts.merge(
    multi_commodity_anomalies,
    on=["State", "District", "Market", "Year"],
    how="inner"
)

display(
    detailed_anomaly_check[
        [
            "State", "District", "Market", "Year",
            "Commodity", "Observations",
            "Previous_Annual_Median",
            "Annual_Median_Price",
            "Yearly_Price_Ratio",
            "Yearly_Price_Change_Pct"
        ]
    ]
    .sort_values(["State", "Market", "Year", "Commodity"])
    .round(2)
)

print("Commodity-level cases:", len(detailed_anomaly_check))

,State,District,Market,Year,Commodity,Observations,Previous_Annual_Median,Annual_Median_Price,Yearly_Price_Ratio,Yearly_Price_Change_Pct
4,Karnataka,Kalburgi,Kalburgi,2022,Bhindi (Ladies Finger),229,265.0,1400.0,5.28,428.30
20,Karnataka,Kalburgi,Kalburgi,2022,Capsicum,225,250.0,1650.0,6.60,560.00
38,Karnataka,Kalburgi,Kalburgi,2022,Green Chilli,234,155.0,1800.0,11.61,1061.29
6,Kerala,Kottayam,Adithyapuram VFPCK,2025,Bitter gourd,96,78.0,6500.0,83.33,8233.33
10,Kerala,Kottayam,Adithyapuram VFPCK,2025,Bottle gourd,15,16.0,1500.0,93.75,9275.00
31,Kerala,Kottayam,Adithyapuram VFPCK,2025,Cowpea (Veg),147,57.0,5300.0,92.98,9198.25
0,Kerala,Palakad,Kollengode,2023,Amaranthus,177,22000.0,3300.0,0.15,-85.00
2,Kerala,Palakad,Kollengode,2023,Beetroot,166,32000.0,2800.0,0.09,-91.25
5,Kerala,Palakad,Kollengode,2023,Bhindi (Ladies Finger),167,35000.0,3300.0,0.09,-90.57
8,Kerala,Palakad,Kollengode,2023,Bitter gourd,180,35000.0,3800.0,0.11,-89.14


Commodity-level cases: 51


In [19]:
# Quantify the impact of multi-commodity market-year anomalies

market_year_impact = comparison_con.execute("""
    WITH annual_prices AS (
        SELECT
            State,
            District,
            Market,
            Commodity,
            YEAR(Arrival_Date) AS Year,
            COUNT(*) AS Observations,
            MEDIAN(Median_Modal_Price) AS Annual_Median_Price
        FROM market_day_prices
        GROUP BY State, District, Market, Commodity, YEAR(Arrival_Date)
    ),

    annual_changes AS (
        SELECT
            *,
            LAG(Annual_Median_Price) OVER (
                PARTITION BY State, District, Market, Commodity
                ORDER BY Year
            ) AS Previous_Annual_Median
        FROM annual_prices
    ),

    market_year_flags AS (
        SELECT
            State,
            District,
            Market,
            Year,
            COUNT(*) FILTER (
                WHERE
                    Previous_Annual_Median > 0
                    AND (
                        Annual_Median_Price / Previous_Annual_Median >= 5
                        OR Annual_Median_Price / Previous_Annual_Median <= 0.2
                    )
            ) AS Affected_Commodities
        FROM annual_changes
        GROUP BY State, District, Market, Year
    )

    SELECT
        COUNT(*) AS Total_Market_Years,
        COUNT(*) FILTER (
            WHERE Affected_Commodities >= 3
        ) AS Multi_Commodity_Anomaly_Groups,
        SUM(Affected_Commodities) AS Total_Affected_Commodity_Groups,
        MAX(Affected_Commodities) AS Maximum_Commodities_Affected
    FROM market_year_flags
""").df()

display(market_year_impact)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Total_Market_Years,Multi_Commodity_Anomaly_Groups,Total_Affected_Commodity_Groups,Maximum_Commodities_Affected
0,7609,11,309.0,15


In [20]:
# Create a reusable register of suspected market-year anomalies

market_year_anomalies = comparison_con.execute("""
    WITH annual_prices AS (
        SELECT
            State,
            District,
            Market,
            Commodity,
            YEAR(Arrival_Date) AS Year,
            COUNT(*) AS Observations,
            MEDIAN(Median_Modal_Price) AS Annual_Median_Price
        FROM market_day_prices
        GROUP BY State, District, Market, Commodity, YEAR(Arrival_Date)
    ),

    annual_changes AS (
        SELECT
            *,
            LAG(Annual_Median_Price) OVER (
                PARTITION BY State, District, Market, Commodity
                ORDER BY Year
            ) AS Previous_Annual_Median
        FROM annual_prices
    ),

    flagged AS (
        SELECT
            *,
            Annual_Median_Price / Previous_Annual_Median
                AS Yearly_Price_Ratio
        FROM annual_changes
        WHERE Previous_Annual_Median > 0
          AND (
              Annual_Median_Price / Previous_Annual_Median >= 5
              OR Annual_Median_Price / Previous_Annual_Median <= 0.2
          )
    )

    SELECT
        State,
        District,
        Market,
        Year,
        COUNT(*) AS Affected_Commodities,
        STRING_AGG(Commodity, ', ' ORDER BY Commodity)
            AS Commodity_List,
        MEDIAN(Yearly_Price_Ratio) AS Median_Price_Ratio,
        MIN(Yearly_Price_Ratio) AS Minimum_Ratio,
        MAX(Yearly_Price_Ratio) AS Maximum_Ratio
    FROM flagged
    GROUP BY State, District, Market, Year
    HAVING COUNT(*) >= 3
    ORDER BY Affected_Commodities DESC
""").df()

print("Suspected market-year anomalies:", len(market_year_anomalies))
display(market_year_anomalies.round(2))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Suspected market-year anomalies: 11


,State,District,Market,Year,Affected_Commodities,Commodity_List,Median_Price_Ratio,Minimum_Ratio,Maximum_Ratio
0,Kerala,Palakad,Kollengode,2023,15,"Amaranthus, Beetroot, Bhindi (Ladies Finger), ...",0.11,0.07,0.19
1,Kerala,Thiruvananthapuram,Pothencode,2025,14,"Amaranthus, Beetroot, Brinjal, Cabbage, Capsic...",0.17,0.11,0.19
2,Kerala,Kottayam,Adithyapuram VFPCK,2025,6,"Amaranthus, Bitter gourd, Bottle gourd, Cowpea...",93.37,51.92,142.86
3,Kerala,Palakad,Kollengode,2024,5,"Cabbage, Carrot, Cowpea (Lobia/Karamani), Pota...",0.10,0.07,0.11
4,Kerala,Thiruvananthapuram,Pothencode,2023,5,"Cabbage, French Beans (Frasbean), Garlic, Ging...",5.48,5.15,10.46
5,Meghalaya,East Khasi Hills,Sohra,2024,5,"Bitter gourd, Brinjal, Cabbage, Carrot, Potato",0.08,0.06,0.14
6,Uttar Pradesh,Chitrakut,Karvi,2023,4,"Bottle gourd, Cabbage, Carrot, Cauliflower",0.10,0.08,0.15
7,Karnataka,Kalburgi,Kalburgi,2022,3,"Bhindi (Ladies Finger), Capsicum, Green Chilli",6.60,5.28,11.61
8,Kerala,Kottayam,Thottuva VFPCK,2025,3,"Bitter gourd, Pumpkin, Tapioca",133.33,55.56,140.79
9,Nagaland,Mokokchung,Mokokchung Town,2023,3,"Bitter gourd, Cabbage, Green Chilli",8.70,0.01,14.29


In [21]:
# Exclude suspected market-year anomalies from primary comparisons

comparison_con.register(
    "market_year_anomaly_register",
    market_year_anomalies
)

comparison_con.execute("""
    CREATE OR REPLACE VIEW market_day_prices_reviewed AS
    SELECT p.*
    FROM market_day_prices p
    WHERE NOT EXISTS (
        SELECT 1
        FROM market_year_anomaly_register a
        WHERE p.State = a.State
          AND p.District = a.District
          AND p.Market = a.Market
          AND YEAR(p.Arrival_Date) = a.Year
    )
""")

reviewed_summary = comparison_con.execute("""
    SELECT
        COUNT(*) AS Remaining_Observations,
        COUNT(DISTINCT Commodity) AS Commodities,
        COUNT(DISTINCT
            State || '|' || District || '|' || Market
        ) AS Unique_Markets
    FROM market_day_prices_reviewed
""").df()

display(reviewed_summary)

print(
    "Excluded observations:",
    comparison_con.execute("""
        SELECT COUNT(*)
        FROM market_day_prices
        WHERE NOT EXISTS (
            SELECT 1
            FROM market_day_prices_reviewed r
            WHERE market_day_prices.Commodity = r.Commodity
              AND market_day_prices.Variety = r.Variety
              AND market_day_prices.Grade = r.Grade
              AND market_day_prices.Arrival_Date = r.Arrival_Date
              AND market_day_prices.State = r.State
              AND market_day_prices.District = r.District
              AND market_day_prices.Market = r.Market
        )
    """).fetchone()[0]
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Remaining_Observations,Commodities,Unique_Markets
0,11385399,45,2222


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Excluded observations: 27382


In [22]:
# Rebuild matched market comparisons using reviewed observations

comparison_con.execute("""
    CREATE OR REPLACE VIEW matched_market_prices_reviewed AS
    WITH reference_prices AS (
        SELECT
            Commodity,
            Variety,
            Grade,
            Arrival_Date,
            MEDIAN(Median_Modal_Price) AS Peer_Median_Price,
            COUNT(*) AS Reporting_Markets
        FROM market_day_prices_reviewed
        GROUP BY Commodity, Variety, Grade, Arrival_Date
        HAVING COUNT(*) >= 3
    )

    SELECT
        p.*,
        r.Peer_Median_Price,
        r.Reporting_Markets,
        100.0 * (
            p.Median_Modal_Price / r.Peer_Median_Price - 1
        ) AS Price_Premium_Pct
    FROM market_day_prices_reviewed p
    INNER JOIN reference_prices r
        USING (Commodity, Variety, Grade, Arrival_Date)
""")

reviewed_match_summary = comparison_con.execute("""
    SELECT
        COUNT(*) AS Matched_Observations,
        COUNT(DISTINCT Commodity) AS Commodities,
        COUNT(DISTINCT State || '|' || District || '|' || Market)
            AS Unique_Markets,
        COUNT(DISTINCT Commodity || '|' || Variety || '|' || Grade)
            AS Commodity_Variety_Grade_Combinations,
        MIN(Arrival_Date) AS First_Date,
        MAX(Arrival_Date) AS Last_Date
    FROM matched_market_prices_reviewed
""").df()

display(reviewed_match_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Matched_Observations,Commodities,Unique_Markets,Commodity_Variety_Grade_Combinations,First_Date,Last_Date
0,11314888,45,2213,283,2021-01-01,2025-10-31


In [ ]:
# Recalculate market performance using reviewed comparisons

reviewed_market_performance = comparison_con.execute("""
    SELECT
        Commodity,
        State,
        District,
        Market,

        COUNT(*) AS Matched_Observations,

        COUNT(DISTINCT Arrival_Date) AS Comparison_Days,

        MEDIAN(Price_Premium_Pct) AS Median_Premium_Pct,

        100.0 * AVG(
            CASE
                WHEN Price_Premium_Pct > 0 THEN 1.0
                ELSE 0.0
            END
        ) AS Pct_Days_Above_Peer_Median,

        MEDIAN(ABS(Price_Premium_Pct))
            AS Median_Absolute_Premium_Pct,

        MEDIAN(Median_Modal_Price) AS Median_Market_Price

    FROM matched_market_prices_reviewed

    GROUP BY Commodity, State, District, Market
""").df()

percentage_columns = [
    "Median_Premium_Pct",
    "Pct_Days_Above_Peer_Median",
    "Median_Absolute_Premium_Pct"
]

reviewed_market_performance[percentage_columns] = (
    reviewed_market_performance[percentage_columns].round(2)
)

display(
    reviewed_market_performance
    .sort_values("Matched_Observations", ascending=False)
    .head(20)
)